In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import stats
import matplotlib.pyplot as plt

In [7]:
#查看格式
df = pd.read_excel('./data/1、2、3.xlsx')

all_feature_id   = np.array(df.columns[1:].tolist())   # ← 加 np.array
all_feature_name = np.array(df.iloc[0, 1:].values)     # ← 加 np.array
data_raw         = df.iloc[1:, 1:].values.astype(np.float64)  # 第1行起第1列起 = 数据
classes          = df.iloc[1:, 0].values.astype(int)        # 第1行起第0列 = 标签


#去掉代谢物Unknown
keep = all_feature_name != 'Unknown'
all_feature_id   = all_feature_id[keep]
all_feature_name = all_feature_name[keep]
data_raw         = data_raw[:, keep]

print(f"样本数: {data_raw.shape[0]}，原始特征数: {data_raw.shape[1]}")
print(f"前5个特征ID:     {all_feature_id[:5]}")
print(f"前5个代谢物名字: {all_feature_name[:5]}")
print(f"标签类别:        {set(classes)}")

样本数: 159，原始特征数: 988
前5个特征ID:     [ 1  4  6 10 11]
前5个代谢物名字: ['2-Butenal, (Z)-'
 '4,5-Dihydrooxazole-5-one, 4-[4-acetoxy-3-methoxybenzylidene]-2-phenyl-'
 'Cyclopropyl methyl carbinol' 'Benzene' 'Ethane, 1,2-dichloro-']
标签类别:        {np.int64(1), np.int64(2), np.int64(3)}


In [8]:
#丰度选择
#丰度选择
mean_data    = np.mean(data_raw, axis=0)
threshold_40 = np.percentile(mean_data, 40)
mask_step1   = mean_data > threshold_40
data_step1   = data_raw[:, mask_step1]

fids_step1_id   = all_feature_id[mask_step1]
fids_step1_name = all_feature_name[mask_step1]

print(f"\nStep1 丰度筛选后特征数: {data_step1.shape[1]}")
print(f"  阈值 P40 = {threshold_40:.2f}")
print(f"  去除特征数: {(~mask_step1).sum()}")


Step1 丰度筛选后特征数: 593
  阈值 P40 = 4181.57
  去除特征数: 395


In [9]:
# Step 2：IQR 筛选
log_step1     = np.log1p(data_step1)
IQR           = (np.percentile(log_step1, 75, axis=0) -
                 np.percentile(log_step1, 25, axis=0))
threshold_iqr = np.percentile(IQR, 25)
mask_step2    = IQR >= threshold_iqr
data_step2    = log_step1[:, mask_step2]

fids_step2_id   = fids_step1_id[mask_step2]
fids_step2_name = fids_step1_name[mask_step2]
IQR_ret         = IQR[mask_step2]
IQR_rem         = IQR[~mask_step2]

print(f"\nStep2 IQR 筛选后特征数: {data_step2.shape[1]}")
print(f"  阈值 P25 = {threshold_iqr:.4f}")
print(f"  去除特征数: {(~mask_step2).sum()}")


Step2 IQR 筛选后特征数: 445
  阈值 P25 = 1.2488
  去除特征数: 148


In [10]:
#保存筛选后的 VOC 数据
col_names = [f"{fid}_{name}" for fid, name in
             zip(fids_step2_id, fids_step2_name)]

df_filtered = pd.DataFrame(data_step2, columns=col_names)
df_filtered.insert(0, 'Class', classes)
df_filtered.to_excel('filtered_voc_step2.xlsx', index=False)

print(f"\n✓ 筛选后数据已保存：filtered_voc_step2.xlsx")
print(f"  样本数: {df_filtered.shape[0]}，特征数: {df_filtered.shape[1] - 1}")
print(f"  前5个特征列名:")
for name in col_names[:5]:
    print(f"    {name}")



✓ 筛选后数据已保存：filtered_voc_step2.xlsx
  样本数: 159，特征数: 445
  前5个特征列名:
    1_2-Butenal, (Z)-
    4_4,5-Dihydrooxazole-5-one, 4-[4-acetoxy-3-methoxybenzylidene]-2-phenyl-
    6_Cyclopropyl methyl carbinol
    10_Benzene
    11_Ethane, 1,2-dichloro-


In [15]:
#自己可以换（建立数据集）
import pandas as pd
import torch
from torch.utils.data import Dataset
from scipy.io import savemat


class VOCDataset(Dataset):
    def __init__(self, path, label_map={1: 0, 2: 0, 3: 1}):
        df = pd.read_excel(path)
        self.y = torch.tensor(df['Class'].map(label_map).values, dtype=torch.long)
        self.X = torch.tensor(df.drop(columns=['Class']).values, dtype=torch.float32)
        self.feat_names = df.columns[1:].tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def to_mat(self, path):
        savemat(path, {
            'X': self.X.numpy(),
            'y': self.y.numpy().reshape(-1, 1),
            'feat_names': self.feat_names,
        })


if __name__ == '__main__':
    ds = VOCDataset('filtered_voc_step2.xlsx')
    ds.to_mat('./data/voc_dataset_1+2_vs_3.mat')
    print(f'samples: {len(ds)}, features: {len(ds.feat_names)}')
    print(f'X: {tuple(ds.X.shape)}, y: {tuple(ds.y.shape)}')
    print('class counts:', torch.bincount(ds.y).tolist())
    print('saved: voc_dataset.mat')

samples: 159, features: 445
X: (159, 445), y: (159,)
class counts: [106, 53]
saved: voc_dataset.mat
